<a href="https://colab.research.google.com/github/unatardedemartes-a11y/ClassFiles/blob/main/Sesion10_275334.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo: Livia Daniela Padilla Barragán**

**Matrícula: 275334**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [107]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [108]:
# Tu código aquí
print(df_marketing.columns)
df_marketing_renombrado = df_marketing.copy()
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.replace(r'(?<=[a-z])(?=[A-Z])', '_', regex=True).str.lower() #decidí agregar un guión bajo entre minuscula y mayuscula porque eran mas columnas las que tendria que renombrar para estandarizar el nombre
df_marketing_renombrado = df_marketing_renombrado.rename(columns={'kidhome':'kid_home','teenhome':'teen_home'})
print()
print(df_marketing_renombrado.columns)


Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')

Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teen_home', 'dt_customer', 'recency', 'mnt_wines', 'mnt_fruits',
       'mnt_meat_products', 'mnt_fish_products', 'mnt_sweet_products',
       'mnt_gold_prods', 'num_deals_purchases', 'num_web_purchases',
       'num_catalog_purchases', 'num_store_purchases', 'num_web_visits_month',
       'accepted_cmp3', 'accepted_cmp4', 'accepted_cmp5', 'accepted_cmp1',
       'accepted_cmp2', 'complai

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [109]:
# Tu código aquí
print(df_netflix.dtypes)
print("Valores nulos en date_added antes de conversión: ",df_netflix['date_added'].isna().sum())
print()
df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed', errors='coerce')
print(df_netflix.dtypes)
print("Valores nulos en date_added despues de conversión: ",df_netflix['date_added'].isna().sum())
print()

show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object
Valores nulos en date_added antes de conversión:  10

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object
Valores nulos en date_added despues de conversión:  10



---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [110]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [111]:
# Tu código aquí
print("Filas duplicadas: ",df_marketing_dup.duplicated().sum())
print("Filas duplicadas por ID: ",df_marketing_dup.duplicated(subset='ID').sum())
df_marketing_dup =df_marketing_dup.drop_duplicates()
print("Filas después de eliminar duplicados: ",len(df_marketing_dup))



Filas duplicadas:  2
Filas duplicadas por ID:  2
Filas después de eliminar duplicados:  2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [112]:
# Tu código aquí
print("Valores faltantes por columna :",df_netflix.isnull().sum())
print()
print("Filas con al menos un valor faltante :",len(df_netflix[df_netflix.isnull().any(axis=1)]))

Valores faltantes por columna : show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Filas con al menos un valor faltante : 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [113]:
# Tu código aquí
completitud = (1 - df_netflix.isnull().sum() / len(df_netflix)) * 100
completitud_con_nulos = completitud[completitud < 100]
print("Porcentaje de completitud (%):",completitud_con_nulos.round(2))
print("Columna con menor completitud: :",{completitud_con_nulos.idxmin()} )


Porcentaje de completitud (%): director      69.32
cast          90.78
country       93.49
date_added    99.87
rating        99.91
dtype: float64
Columna con menor completitud: : {'director'}


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [114]:
# Tu código aquí
print("Valores de columna Marital_Status",df_marketing['Marital_Status'].value_counts())
print()
df_marketing.loc[df_marketing['Marital_Status'] == 'Alone', 'Marital_Status'] = 'Single'
df_marketing.loc[(df_marketing['Marital_Status'] == 'Absurd') | (df_marketing['Marital_Status'] == 'YOLO'), 'Marital_Status'] = 'Prefer Not to Respond'
print("Valores de columna Marital_Status despues de remplazar valores",df_marketing['Marital_Status'].value_counts())

Valores de columna Marital_Status Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

Valores de columna Marital_Status despues de remplazar valores Marital_Status
Married                  864
Together                 580
Single                   483
Divorced                 232
Widow                     77
Prefer Not to Respond      4
Name: count, dtype: int64


1.   Clasificaria Alone a Single porque basicamente es lo mismo
2.   Remplazaria los valores de YOLO y Absurd al valor de ''Prefer Not to Respond' porque significa que un porcentaje de los entrevistados no consideran relevante o les incomoda la pregunta.


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [115]:
# Tu código aquí
cumple = df_netflix['show_id'].str.match(r'^s\d+$') #si encuentra el patron que se busca regresa un valor de True\
print("Porcentaje de cumplimiento: ",cumple.mean() * 100, "%")


Porcentaje de cumplimiento:  100.0 %


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [116]:
# Tu código aquí
print("Describe:  ",df_marketing[['Year_Birth']].describe())
print()
print("Filas con el año de nacimiento más antíguo")
print(df_marketing.nsmallest(10, 'Year_Birth')[['ID', 'Year_Birth']])

Describe:           Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000

Filas con el año de nacimiento más antíguo
         ID  Year_Birth
239   11004        1893
339    1150        1899
192    7829        1900
1950   6663        1940
424    6932        1941
39     2968        1943
358    6142        1943
415    7106        1943
894    8800        1943
1150   1453        1943


Sí, son errores de captura, porque un año de nacimiento entr 1893 y 1900 implicaría una edad de mas de 120 años lo que la hace fuera del rango biológicamente posible

---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [117]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [118]:
# Paso 1 — ajuste de tipos
print(df_practica.dtypes)
print()
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
print("Tipos de columnas despues de convertir")
print(df_practica.dtypes)

ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object

Tipos de columnas despues de convertir
ID                       int64
Year_Birth               int64
Education     

In [119]:
# Paso 2 — duplicados
print("Filas duplicadas: ",df_practica.duplicated().sum())
df_practica =df_practica.drop_duplicates()
print("Filas después de eliminar duplicados: ",len(df_practica))


Filas duplicadas:  1
Filas después de eliminar duplicados:  15


In [120]:
# Paso 3 — valores faltantes
print("Valores faltantes por columna :",df_practica.isnull().sum())


Valores faltantes por columna : ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [121]:
# Paso 4 — exploración categórica
print("Valores únicos de Marital_Status:")
print(df_practica['Marital_Status'].unique())
#Si, los valores son aceptables, Together significa convivencia en pareja pero sin matrimonio legal

Valores únicos de Marital_Status:
['Together' 'Single' 'Married' 'Divorced']


**Tu reporte de profiling:**

El dataset contenia 16 filas con 3 columnas, se convirtió la columna Income a numérico con errors='coerce', dejando "sesenta mil" como NaN (1 nulo). Se eliminaron filas duplicadas (ID 4557 repetido). El dataset queda limpio, con 1 valor faltante en Income y sin duplicados



